# Direction 1 v9 — VQ-Only VFL Privacy Study

**Supersedes v6/v7/v8.** Implements the plan in `v9_notebook_plan.md`.

## What changed vs v8
- **No GRL.** All `grad_reverse` / `V8LabelAdversary` / `dann_lambda` plumbing is removed.
- **No label-inference attack.** Privacy is measured only by reconstruction (SSIM + LPIPS).
  Label inference is structurally unsuppressable in VFL without cryptographic methods —
  the task loss forces class-discriminative representations — so suppressing it is not
  the right privacy claim.
- **Train everything from scratch** (no checkpoint loading from v6/v7/v8).
- **11 ablations × 2 seeds = 22 training runs.** Two new vs the v1 plan: `H_vq_no_kmeans`
  (codebook-init ablation) and `S_rand_sign` (any-binary-code baseline).
- **Reconstruction eval on all 11 methods** (v8 only ran it on V8_main).
- **Stronger InverNet:** ResBlocks at the bottleneck, refinement conv, MSE + 0.1·LPIPS
  loss, 30 epochs for all methods (testing).
- **Aligned 6-sample reconstruction grid** across all methods for visual comparison.

## Stages (checkpoint-based; each is independently re-runnable)
- **Stage A:** train 22 passive/active pairs; save checkpoint + per-epoch history.
- **Stage B:** load each checkpoint, train InverNetV9, save SSIM/LPIPS + grid PNG.
- **Stage C:** aggregate JSONs → results table, Pareto curve, multi-row recon figure.

The 22 training runs likely require 2 Kaggle GPU sessions for Stage A. Stage A skips
runs whose checkpoints already exist in `CKPT_DIR`, so re-running just resumes.


In [1]:
!ls "/kaggle/input/datasets/zarishnadeem/vfl-v9-shaz/results_d1v9/checkpoints"

A_plain_vfl_seed42.pt  H_vq_K256_seed42.pt    S_rand_sign_seed43.pt
A_plain_vfl_seed43.pt  H_vq_K64_seed42.pt     S_sign_quant_seed42.pt
A_proj_vfl_seed42.pt   H_vq_K64_seed43.pt     S_sign_quant_seed43.pt
A_proj_vfl_seed43.pt   S_rand_sign_seed42.pt


In [2]:
import os
import shutil

# ── FILL IN your Kaggle dataset path below (check the Data sidebar) ──
DATASET_ROOT = "/kaggle/input/YOUR-DATASET-SLUG/results_final"

source_ckpt_dir    = f"{DATASET_ROOT}/checkpoints"
target_ckpt_dir    = "/kaggle/working/results_d1v9/checkpoints"
target_results_dir = "/kaggle/working/results_d1v9"

os.makedirs(target_ckpt_dir, exist_ok=True)
os.makedirs(target_results_dir, exist_ok=True)

if not os.path.exists(DATASET_ROOT):
    print(f"❌ Dataset not found at {DATASET_ROOT}. Check Kaggle data sidebar.")
else:
    # Copy result_*.json from the root of results_final/ into RESULTS_DIR
    copied_jsons = 0
    for filename in os.listdir(DATASET_ROOT):
        if filename.endswith(".json"):
            shutil.copy(os.path.join(DATASET_ROOT, filename), target_results_dir)
            copied_jsons += 1
    print(f"Copied {copied_jsons} result JSONs.")

    # Copy .pt checkpoints from results_final/checkpoints/
    copied_ckpts = 0
    for filename in os.listdir(source_ckpt_dir):
        if filename.endswith(".pt"):
            shutil.copy(os.path.join(source_ckpt_dir, filename), target_ckpt_dir)
            copied_ckpts += 1
    print(f"✅ Copied {copied_ckpts} checkpoints.")
    print("Checkpoints:", sorted(os.listdir(target_ckpt_dir)))

Copying files...


✅ Success! Copied 11 checkpoint files.
Files currently in working checkpoints directory: ['S_rand_sign_seed42.pt', 'A_proj_vfl_seed42.pt', 'A_plain_vfl_seed43.pt', 'H_vq_K64_seed42.pt', 'H_vq_K64_seed43.pt', 'A_plain_vfl_seed42.pt', 'S_sign_quant_seed43.pt', 'A_proj_vfl_seed43.pt', 'H_vq_K256_seed42.pt', 'S_rand_sign_seed43.pt', 'S_sign_quant_seed42.pt']


In [3]:
# Install runtime dependencies if missing. Safe on Kaggle.
import subprocess, sys
for pkg in ['lpips', 'scikit-image', 'timm']:
    mod = pkg.replace('-', '_').replace('scikit_image', 'skimage')
    try:
        __import__(mod)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Environment ready.")


Installing lpips ...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.1 MB/s eta 0:00:00


Environment ready.


In [4]:
import os, json, math, time, copy, hashlib, traceback
import numpy as np
import pandas as pd
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple, Any
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms

from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.cluster import KMeans

import timm

print(f"torch {torch.__version__}  cuda_available={torch.cuda.is_available()}")


torch 2.10.0+cu128  cuda_available=True


In [5]:
# -------- Paths (Kaggle defaults; override locally if needed) --------
IMG_DIR     = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
GT_CSV      = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_GroundTruth.csv"
META_CSV    = "/kaggle/input/datasets/andrewmvd/isic-2019/ISIC_2019_Training_Metadata.csv"
CACHE_PATH  = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/images_256.npy"
FOLD_PATH   = "/kaggle/input/datasets/taimurjahanzeb/isic-2019-256-cache/fold_indices.npz"
RESULTS_DIR = "/kaggle/working/results_d1v9"
CKPT_DIR    = f"{RESULTS_DIR}/checkpoints"
FIG_DIR     = f"{RESULTS_DIR}/figures"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


# -------- Stage toggles --------
# Set False on a "Run All" if you want to skip a stage. Each stage is idempotent
# (skips already-completed runs via checkpoints / saved JSONs) so leaving them
# True is the intended Kaggle workflow.
RUN_STAGE_A = True   # Train all (method, seed) pairs
RUN_STAGE_B = True   # Reconstruction privacy evaluation
RUN_STAGE_C = True   # Aggregation, Pareto curve, visual comparison


@dataclass
class V9Config:
    # Data
    input_size: int       = 224
    store_size: int       = 256
    batch_size: int       = 32
    fold: int             = 0
    # Training
    lr: float             = 1e-4
    weight_decay: float   = 1e-5
    warmup_epochs: int    = 5
    stage1_epochs: int    = 30
    patience: int         = 10
    # Quantizer (per-method overrides supplied via MethodSpec)
    vq_proj_dim: int      = 128
    vq_num_subspaces: int = 8
    vq_codebook_size: int = 256
    vq_commitment_weight: float = 0.25
    vq_codebook_weight: float   = 1.0
    vq_warmup_epochs: int = 3
    vq_use_kmeans_init: bool = True
    kmeans_samples: int   = 4096
    # Baseline VFL noise (kept identical to v6/v7/v8 for fair comparison)
    rm_sigma_fwd: float   = 0.01
    rm_sigma_bwd: float   = 0.01
    # Usage entropy regulariser (label-free; prevents codebook collapse)
    usage_entropy_weight: float = 0.01
    # Execution
    seeds: Tuple[int, ...] = (42,43)
    num_workers: int      = 2

CFG = V9Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Config: {asdict(CFG)}")

with open(f"{RESULTS_DIR}/config.json", 'w') as f:
    json.dump(asdict(CFG), f, indent=2)

assert CFG.vq_proj_dim % CFG.vq_num_subspaces == 0


Device: cuda
Config: {'input_size': 224, 'store_size': 256, 'batch_size': 32, 'fold': 0, 'lr': 0.0001, 'weight_decay': 1e-05, 'warmup_epochs': 5, 'stage1_epochs': 30, 'patience': 10, 'vq_proj_dim': 128, 'vq_num_subspaces': 8, 'vq_codebook_size': 256, 'vq_commitment_weight': 0.25, 'vq_codebook_weight': 1.0, 'vq_warmup_epochs': 3, 'vq_use_kmeans_init': True, 'kmeans_samples': 4096, 'rm_sigma_fwd': 0.01, 'rm_sigma_bwd': 0.01, 'usage_entropy_weight': 0.01, 'seeds': (42, 43), 'num_workers': 2}


In [6]:
# -------- Ground truth, metadata, folds --------
gt = pd.read_csv(GT_CSV)
gt = gt.drop(columns=['UNK'], errors='ignore')
CLASS_NAMES = gt.columns[1:].tolist()
NUM_CLASSES = len(CLASS_NAMES)

meta_df = pd.read_csv(META_CSV)
merged  = gt.merge(meta_df, on='image', how='inner')

bins       = list(range(0, 90, 5))
age_labels = [f"age_{b}_{b+4}" for b in bins[:-1]] + ["age_85plus"]
merged['age_bin'] = pd.cut(
    merged['age_approx'], bins=bins + [np.inf], labels=age_labels, right=False
).astype(str)
merged.loc[merged['age_approx'].isna(), 'age_bin'] = 'age_unknown'
age_oh = pd.get_dummies(merged['age_bin']).astype(np.float32).values

site_col = ('anatom_site_general_challenge'
            if 'anatom_site_general_challenge' in merged.columns
            else 'anatom_site_general')
merged[site_col] = merged[site_col].fillna('unknown')
loc_oh = pd.get_dummies(merged[site_col]).astype(np.float32).values

merged['sex'] = merged['sex'].fillna('unknown')
sex_oh = pd.get_dummies(merged['sex']).astype(np.float32).values

meta_array   = np.concatenate([age_oh, loc_oh, sex_oh], axis=1).astype(np.float32)
META_DIM     = meta_array.shape[1]
labels_array = merged[CLASS_NAMES].values.astype(np.float32)
label_ints   = np.argmax(labels_array, axis=1)

class_counts         = np.bincount(label_ints, minlength=NUM_CLASSES).astype(np.float32)
class_weights        = 1.0 / class_counts
class_weights        = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

tail_classes = sorted(np.argsort(class_counts)[:4].tolist())

assert os.path.exists(FOLD_PATH),  f"fold file missing: {FOLD_PATH}"
assert os.path.exists(CACHE_PATH), f"image cache missing: {CACHE_PATH}"
_fd = np.load(FOLD_PATH)
train_ind = _fd[f'train_{CFG.fold}']
val_ind   = _fd[f'val_{CFG.fold}']
all_images = np.load(CACHE_PATH)

print("=" * 72)
print(f"  Dataset:        ISIC-2019 (8 classes; UNK dropped)")
print(f"  Fold:           {CFG.fold}")
print(f"  Train samples:  {len(train_ind)}")
print(f"  Val samples:    {len(val_ind)}")
print(f"  Classes:        {CLASS_NAMES}")
print(f"  Class counts:   {class_counts.astype(int).tolist()}")
print(f"  Tail (4 rare): {[CLASS_NAMES[c] for c in tail_classes]}")
print(f"  Metadata dim:   {META_DIM}")
print("=" * 72)

with open(f"{RESULTS_DIR}/dataset_audit.json", 'w') as f:
    json.dump({
        'dataset': 'ISIC-2019',
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'class_counts': class_counts.astype(int).tolist(),
        'tail_class_indices': tail_classes,
        'tail_class_names': [CLASS_NAMES[c] for c in tail_classes],
        'fold': CFG.fold,
        'n_train': int(len(train_ind)),
        'n_val': int(len(val_ind)),
        'meta_dim': int(META_DIM),
    }, f, indent=2)


  Dataset:        ISIC-2019 (8 classes; UNK dropped)
  Fold:           0
  Train samples:  20264
  Val samples:    5067
  Classes:        ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
  Class counts:   [4522, 12875, 3323, 867, 2624, 239, 253, 628]
  Tail (4 rare): ['AK', 'DF', 'VASC', 'SCC']
  Metadata dim:   31


In [7]:
SET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
SET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CFG.input_size, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=32./255., saturation=0.5),
    transforms.ToTensor(), transforms.Normalize(SET_MEAN, SET_STD),
])
val_transforms = transforms.Compose([
    transforms.CenterCrop(CFG.input_size),
    transforms.ToTensor(), transforms.Normalize(SET_MEAN, SET_STD),
])

class ISICDataset(Dataset):
    def __init__(self, images, labels, meta, transform):
        self.images = images
        self.labels = labels
        self.meta = meta
        self.transform = transform
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        label = torch.tensor(np.argmax(self.labels[idx]), dtype=torch.long)
        meta  = torch.tensor(self.meta[idx])
        img   = self.transform(Image.fromarray(self.images[idx]))
        return img, meta, label

def make_loaders(train_ind, val_ind, batch_size=None, shuffle_train=True):
    bs = batch_size or CFG.batch_size
    train_ds = ISICDataset(all_images[train_ind], labels_array[train_ind],
                           meta_array[train_ind], train_transforms)
    val_ds   = ISICDataset(all_images[val_ind], labels_array[val_ind],
                           meta_array[val_ind], val_transforms)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=shuffle_train,
                              num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=bs, shuffle=False,
                              num_workers=CFG.num_workers, pin_memory=True)
    return train_loader, val_loader

print("Datasets and loaders defined.")


Datasets and loaders defined.


## §3 — Model definitions

Reused from v8 (stable names): `V8Backbone`, `V8Projection`, `V8ProductQuantizer`,
`V8MetadataMLP`, `V8ActiveClient`. New passive clients for v9: `PlainPassiveClient`,
`ProjPassiveClient`, `SignPassiveClient`, `RandSignPassiveClient`, `VQPassiveClient`.

All passive clients share the same forward signature so the training loop is uniform:
```
emb, indices_or_None, vq_loss_or_zero, z_raw = passive(x, use_quantizer=...)
```


In [8]:
class V8Backbone(nn.Module):
    '''EfficientNet-B0 image encoder -> 1280-d continuous embedding.'''
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.net = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        self.out_dim = 1280
    def forward(self, x):
        return self.net(x)


class V8Projection(nn.Module):
    '''Linear(1280 -> proj_dim) + BatchNorm1d. Same shape as v6/v7/v8.'''
    def __init__(self, in_dim: int = 1280, out_dim: int = 128):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.bn     = nn.BatchNorm1d(out_dim)
    def forward(self, h):
        return self.bn(self.linear(h))


In [9]:
class V8ProductQuantizer(nn.Module):
    '''Product quantization of D-dim vectors into M subspaces of size D/M.

    forward(z) -> (q, indices, vq_loss). q is straight-through; indices are (B, M).
    K-means seeding via kmeans_init_from_batch (label-free; collected on a train subset).
    '''
    def __init__(self, proj_dim: int, num_subspaces: int, codebook_size: int,
                 commitment_weight: float = 0.25):
        super().__init__()
        assert proj_dim % num_subspaces == 0
        self.proj_dim = proj_dim
        self.M = num_subspaces
        self.K = codebook_size
        self.subdim = proj_dim // num_subspaces
        self.commitment_weight = commitment_weight
        self.codebooks = nn.Parameter(
            torch.randn(num_subspaces, codebook_size, self.subdim) * 0.02
        )
        self.register_buffer('usage_counts',
            torch.zeros(num_subspaces, codebook_size, dtype=torch.long))

    def forward(self, z: torch.Tensor):
        B, D = z.shape
        assert D == self.proj_dim
        z_sub = z.view(B, self.M, self.subdim)
        distances = (
            z_sub.unsqueeze(2).pow(2).sum(-1)
            + self.codebooks.pow(2).sum(-1).unsqueeze(0)
            - 2.0 * torch.einsum('bmd,mkd->bmk', z_sub, self.codebooks)
        )
        indices = distances.argmin(dim=-1)
        q_sub = torch.gather(
            self.codebooks.unsqueeze(0).expand(B, -1, -1, -1),
            2,
            indices.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, self.subdim),
        ).squeeze(2)
        codebook_loss   = F.mse_loss(q_sub, z_sub.detach())
        commitment_loss = F.mse_loss(z_sub, q_sub.detach())
        vq_loss = codebook_loss + self.commitment_weight * commitment_loss
        q_sub_st = z_sub + (q_sub - z_sub).detach()
        q = q_sub_st.reshape(B, D)
        with torch.no_grad():
            for m in range(self.M):
                counts = torch.bincount(indices[:, m], minlength=self.K)
                self.usage_counts[m] += counts
        return q, indices, vq_loss

    @torch.no_grad()
    def reset_usage(self):
        self.usage_counts.zero_()

    @torch.no_grad()
    def kmeans_init_from_batch(self, z_samples: torch.Tensor, seed: int = 0):
        '''K-means++ seeding per subspace. Label-free.'''
        N = z_samples.size(0)
        assert N >= self.K
        z_np_sub = z_samples.detach().cpu().numpy().reshape(N, self.M, self.subdim)
        new_cb = np.zeros((self.M, self.K, self.subdim), dtype=np.float32)
        for m in range(self.M):
            km = KMeans(n_clusters=self.K, init='k-means++',
                        n_init=3, max_iter=50, random_state=seed + m)
            km.fit(z_np_sub[:, m, :])
            new_cb[m] = km.cluster_centers_.astype(np.float32)
        self.codebooks.data.copy_(torch.from_numpy(new_cb).to(self.codebooks.device))
        self.reset_usage()


In [10]:
class V8MetadataMLP(nn.Module):
    def __init__(self, meta_dim: int, out_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, out_dim),
            nn.ReLU(inplace=True),
        )
    def forward(self, m):
        return self.net(m)


class V8ActiveClient(nn.Module):
    '''Receives transmitted representation + metadata, outputs class logits.'''
    def __init__(self, passive_emb_dim: int, meta_dim: int, num_classes: int):
        super().__init__()
        self.meta_mlp   = V8MetadataMLP(meta_dim, out_dim=128)
        self.classifier = nn.Sequential(
            nn.Linear(passive_emb_dim + 128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, emb, meta):
        m = self.meta_mlp(meta)
        return self.classifier(torch.cat([emb, m], dim=1))


In [11]:
# ---------------- Passive clients (5 types) ----------------
# Common forward signature:
#   emb, indices_or_None, vq_loss_or_zero, z_raw = passive(x, use_quantizer=True)
# - emb     : the transmitted representation (what the active side receives)
# - z_raw   : pre-quantization projection (used for soft usage entropy on VQ methods;
#             equal to emb for non-VQ methods)
# - indices : codebook indices (only for VQ); None otherwise
# - vq_loss : codebook + commitment loss (only for VQ); 0 otherwise

class PlainPassiveClient(nn.Module):
    '''A_plain_vfl: backbone only, emits 1280-d continuous.'''
    def __init__(self, cfg: V9Config):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        self.passive_emb_dim = self.backbone.out_dim  # 1280
    def forward(self, x, use_quantizer: bool = True):
        h = self.backbone(x)
        return h, None, torch.zeros((), device=h.device), h


class ProjPassiveClient(nn.Module):
    '''A_proj_vfl: backbone + projection, emits 128-d continuous.'''
    def __init__(self, cfg: V9Config):
        super().__init__()
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.passive_emb_dim = cfg.vq_proj_dim  # 128
    def forward(self, x, use_quantizer: bool = True):
        h = self.backbone(x)
        z = self.projection(h)
        return z, None, torch.zeros((), device=z.device), z


class SignPassiveClient(nn.Module):
    '''S_sign_quant: backbone + learned projection + sign() with STE. 128 bits.'''
    def __init__(self, cfg: V9Config):
        super().__init__()
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.passive_emb_dim = cfg.vq_proj_dim
    def forward(self, x, use_quantizer: bool = True):
        h = self.backbone(x)
        z = self.projection(h)
        s = z.sign()
        s_ste = z + (s - z).detach()
        return s_ste, None, torch.zeros((), device=z.device), z


class RandSignPassiveClient(nn.Module):
    '''S_rand_sign: backbone + FROZEN random projection + sign() with STE. 128 bits.

    The projection W is a buffer (in state_dict, not a Parameter) so it never
    receives gradient updates. Backbone trains, classifier trains, projection
    stays frozen. With W ~ N(0, 1/sqrt(in_dim)) each row has zero mean, so
    sign(W @ h) is approximately balanced even though h = backbone(x) is post-ReLU.
    '''
    def __init__(self, cfg: V9Config, init_seed: int = 0):
        super().__init__()
        self.backbone = V8Backbone(pretrained=True)
        in_dim  = self.backbone.out_dim
        out_dim = cfg.vq_proj_dim
        gen = torch.Generator().manual_seed(int(init_seed))
        W = torch.randn(in_dim, out_dim, generator=gen) / (in_dim ** 0.5)
        self.register_buffer('proj_W', W)
        self.passive_emb_dim = out_dim

    def forward(self, x, use_quantizer: bool = True):
        h = self.backbone(x)
        z = h @ self.proj_W   # (B, out_dim)
        s = z.sign()
        s_ste = z + (s - z).detach()
        return s_ste, None, torch.zeros((), device=z.device), z


class VQPassiveClient(nn.Module):
    '''H_vq_*: backbone + projection + product quantizer.'''
    def __init__(self, cfg: V9Config,
                 codebook_size: int = None,
                 num_subspaces: int = None,
                 commitment_weight: float = None):
        super().__init__()
        K = codebook_size       if codebook_size       is not None else cfg.vq_codebook_size
        M = num_subspaces       if num_subspaces       is not None else cfg.vq_num_subspaces
        beta = commitment_weight if commitment_weight  is not None else cfg.vq_commitment_weight
        self.backbone   = V8Backbone(pretrained=True)
        self.projection = V8Projection(self.backbone.out_dim, cfg.vq_proj_dim)
        self.quantizer  = V8ProductQuantizer(
            proj_dim=cfg.vq_proj_dim,
            num_subspaces=M,
            codebook_size=K,
            commitment_weight=beta,
        )
        self.passive_emb_dim = cfg.vq_proj_dim

    def forward(self, x, use_quantizer: bool = True):
        h = self.backbone(x)
        z = self.projection(h)
        if not use_quantizer:
            return z, None, torch.zeros((), device=z.device), z
        q, idx, vq_loss = self.quantizer(z)
        return q, idx, vq_loss, z

    @torch.no_grad()
    def collect_projections(self, loader, n_batches: int = 16):
        was_training = self.training
        self.eval()
        zs = []
        count = 0
        for images, _meta, _labels in loader:
            images = images.to(next(self.parameters()).device)
            h = self.backbone(images)
            z = self.projection(h)
            zs.append(z.detach())
            count += 1
            if count >= n_batches:
                break
        self.train(was_training)
        return torch.cat(zs, dim=0)


print("Passive client classes defined.")


Passive client classes defined.


## §4 — Training utilities

`set_seed`, `apply_random_mask` (forward/backward channel noise — kept for v6/v7/v8
parity), `evaluate_utility`, `save_metrics`, checkpoint load helpers, and
`usage_entropy_loss` (soft-assignment entropy regulariser, label-free).


In [12]:
def set_seed(seed: int):
    import random as _random
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def apply_random_mask(x: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0:
        return x
    return x + torch.randn_like(x) * sigma


@torch.no_grad()
def evaluate_utility(passive_model, active_model, val_loader,
                     use_quantizer: bool = True) -> Dict[str, Any]:
    passive_model.eval(); active_model.eval()
    all_probs, all_labels = [], []
    for images, meta, labels in val_loader:
        images, meta = images.to(DEVICE), meta.to(DEVICE)
        emb, _, _, _ = passive_model(images, use_quantizer=use_quantizer)
        logits = active_model(emb, meta)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = np.argmax(all_probs, axis=1)
    wacc = float(balanced_accuracy_score(all_labels, preds))
    labels_oh = np.eye(NUM_CLASSES)[all_labels.astype(int)]
    try:
        auc = float(roc_auc_score(labels_oh, all_probs, average='macro', multi_class='ovr'))
    except ValueError:
        auc = float('nan')
    pcr = []
    for c in range(NUM_CLASSES):
        mask = all_labels == c
        pcr.append(float((preds[mask] == c).mean()) if mask.sum() > 0 else 0.0)
    tail_mean = float(np.mean([pcr[c] for c in tail_classes]))
    return {
        'val_wacc': wacc, 'val_auc': auc,
        'per_class_recall': pcr, 'tail_mean': tail_mean,
        'n_val': int(len(all_labels)),
    }


def save_metrics(path: str, obj: Dict):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def _torch_load(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


def usage_entropy_loss(quantizer: 'V8ProductQuantizer', z: torch.Tensor,
                       tau: float = 1.0) -> torch.Tensor:
    '''Differentiable entropy regulariser via soft assignment over pre-quant z.
    Minimising this maximises code-usage entropy, preventing collapse. Label-free.
    '''
    B, D = z.shape
    z_sub = z.view(B, quantizer.M, quantizer.subdim)
    distances = (
        z_sub.unsqueeze(2).pow(2).sum(-1)
        + quantizer.codebooks.pow(2).sum(-1).unsqueeze(0)
        - 2.0 * torch.einsum('bmd,mkd->bmk', z_sub, quantizer.codebooks)
    )
    p_soft = F.softmax(-distances / tau, dim=-1)
    p_mean = p_soft.mean(dim=0)
    eps = 1e-10
    ent = -(p_mean * (p_mean + eps).log()).sum(dim=1).mean()
    return -ent

print("Training utilities defined.")


Training utilities defined.


In [13]:
# -------- Passive client factory --------
def build_passive(spec, seed: int):
    '''Build the passive client for a given MethodSpec.'''
    cfg = CFG
    if spec.passive_type == 'plain':
        return PlainPassiveClient(cfg)
    if spec.passive_type == 'proj':
        return ProjPassiveClient(cfg)
    if spec.passive_type == 'sign':
        return SignPassiveClient(cfg)
    if spec.passive_type == 'rand_sign':
        # Use seed for reproducible random projection
        return RandSignPassiveClient(cfg, init_seed=seed)
    if spec.passive_type == 'vq':
        return VQPassiveClient(
            cfg,
            codebook_size=spec.vq_codebook_size,
            num_subspaces=spec.vq_num_subspaces,
            commitment_weight=spec.vq_commitment_weight,
        )
    raise ValueError(f"unknown passive_type: {spec.passive_type}")


In [14]:
def train_v9(spec, seed: int) -> Dict[str, Any]:
    '''Single training run for one MethodSpec at one seed.

    Saves: ${CKPT_DIR}/{name}_seed{seed}.pt and ${RESULTS_DIR}/history_{name}_seed{seed}.json
    Returns: result dict with best_wacc, ckpt_path, comm_bits, etc.
    '''
    cfg = CFG
    set_seed(seed)
    print(f"\n{'=' * 80}")
    print(f"  Training: {spec.name}  seed={seed}  passive_type={spec.passive_type}")
    print(f"  comm_bits={spec.comm_bits}", end='')
    if spec.passive_type == 'vq':
        print(f"  K={spec.vq_codebook_size}  M={spec.vq_num_subspaces}"
              f"  beta={spec.vq_commitment_weight}  kmeans_init={spec.vq_use_kmeans_init}",
              end='')
    print()
    print(f"{'=' * 80}")

    train_loader, val_loader = make_loaders(train_ind, val_ind)

    passive = build_passive(spec, seed=seed).to(DEVICE)
    active = V8ActiveClient(
        passive_emb_dim=passive.passive_emb_dim,
        meta_dim=META_DIM,
        num_classes=NUM_CLASSES,
    ).to(DEVICE)

    # Trainable params on the passive side. RandSignPassiveClient's proj_W is a
    # buffer and so is automatically excluded by .parameters().
    opt_passive = optim.Adam(
        [p for p in passive.parameters() if p.requires_grad],
        lr=cfg.lr, weight_decay=cfg.weight_decay,
    )
    opt_active = optim.Adam(active.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    n_sched_steps = max(spec.train_epochs - cfg.warmup_epochs, 1)
    sched_passive = CosineAnnealingLR(opt_passive, T_max=n_sched_steps, eta_min=1e-6)
    sched_active  = CosineAnnealingLR(opt_active,  T_max=n_sched_steps, eta_min=1e-6)

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    is_vq = (spec.passive_type == 'vq')
    history = []
    best_wacc = 0.0
    best_pcr = [0.0] * NUM_CLASSES
    best_epoch = 0
    patience_count = 0
    ckpt_path = f"{CKPT_DIR}/{spec.name}_seed{seed}.pt"

    for epoch in range(1, spec.train_epochs + 1):
        passive.train(); active.train()
        if is_vq:
            passive.quantizer.reset_usage()

        # VQ curriculum: continuous warmup, then quantizer on
        use_vq_this_epoch = is_vq and (epoch > cfg.vq_warmup_epochs)

        # K-means codebook seeding right after VQ warmup ends.
        # Collect at least 4 * K samples per subspace so K-means is non-degenerate.
        if is_vq and spec.vq_use_kmeans_init and (epoch == cfg.vq_warmup_epochs + 1):
            print(f"  [epoch {epoch}] running K-means++ codebook seeding ...")
            target_samples = max(cfg.kmeans_samples, spec.vq_codebook_size * 4)
            n_batches_needed = max(8, math.ceil(target_samples / cfg.batch_size))
            z_for_init = passive.collect_projections(train_loader, n_batches=n_batches_needed)
            if z_for_init.size(0) > target_samples:
                idxs = torch.randperm(z_for_init.size(0))[:target_samples]
                z_for_init = z_for_init[idxs]
            passive.quantizer.kmeans_init_from_batch(z_for_init, seed=seed)
            print(f"  [epoch {epoch}] K-means done, {z_for_init.size(0)} samples used "
                  f"(K={spec.vq_codebook_size}, M={spec.vq_num_subspaces})")

        # LR warmup
        if epoch <= cfg.warmup_epochs:
            lr_mul = epoch / cfg.warmup_epochs
            for pg in opt_passive.param_groups: pg['lr'] = cfg.lr * lr_mul
            for pg in opt_active.param_groups:  pg['lr'] = cfg.lr * lr_mul

        ep_task, ep_vq, ep_use, n_batches = 0.0, 0.0, 0.0, 0
        preds_list, lbls_list = [], []

        for images, meta, labels in train_loader:
            images = images.to(DEVICE); meta = meta.to(DEVICE); labels = labels.to(DEVICE)

            # ----- Passive forward (no labels) -----
            emb, indices, vq_loss, z_raw = passive(images, use_quantizer=use_vq_this_epoch)
            if not use_vq_this_epoch:
                vq_loss = torch.zeros((), device=DEVICE)

            # ----- Channel transmit (forward noise; v6/v7/v8 parity) -----
            pred_masked = apply_random_mask(emb.detach(), cfg.rm_sigma_fwd)
            emb_received = pred_masked.requires_grad_(True)

            # ----- Active forward + backward -----
            opt_active.zero_grad()
            logits = active(emb_received, meta)
            task_loss = criterion(logits, labels)
            task_loss.backward()
            grad_to_passive = emb_received.grad.clone()
            opt_active.step()

            # ----- Channel return (backward noise) -----
            grad_masked = apply_random_mask(grad_to_passive, cfg.rm_sigma_bwd)

            # ----- Passive backward -----
            opt_passive.zero_grad()
            usage_reg = torch.zeros((), device=DEVICE)
            if use_vq_this_epoch and cfg.usage_entropy_weight > 0 and z_raw is not None:
                usage_reg = usage_entropy_loss(passive.quantizer, z_raw)
            total_passive = (
                (emb * grad_masked).sum()
                + cfg.vq_codebook_weight * vq_loss
                + cfg.usage_entropy_weight * usage_reg
            )
            total_passive.backward()
            opt_passive.step()

            ep_task += float(task_loss.item())
            ep_vq   += float(vq_loss.item() if torch.is_tensor(vq_loss) else vq_loss)
            ep_use  += float(usage_reg.item() if torch.is_tensor(usage_reg) else usage_reg)
            n_batches += 1
            preds_list.append(logits.detach().cpu())
            lbls_list.append(labels.cpu())

        if epoch > cfg.warmup_epochs:
            sched_passive.step(); sched_active.step()

        val_metrics = evaluate_utility(
            passive, active, val_loader, use_quantizer=use_vq_this_epoch,
        )

        # Codebook usage stats
        if is_vq:
            usage = passive.quantizer.usage_counts.detach().cpu().numpy()
            active_fraction = float((usage > 0).mean()) if use_vq_this_epoch else float('nan')
        else:
            active_fraction = float('nan')

        train_preds  = torch.cat(preds_list).argmax(dim=1).numpy()
        train_labels = torch.cat(lbls_list).numpy()
        train_wacc   = float(balanced_accuracy_score(train_labels, train_preds))

        avg_task = ep_task / max(1, n_batches)
        avg_vq   = ep_vq   / max(1, n_batches)
        avg_use  = ep_use  / max(1, n_batches)
        vq_flag = '(VQ on)' if use_vq_this_epoch else ('(cont.)' if is_vq else '')
        print(f"Ep {epoch:02d}/{spec.train_epochs} {vq_flag} | "
              f"L_task={avg_task:.4f} L_vq={avg_vq:.4f} L_use={avg_use:.4f} | "
              f"Tr={train_wacc:.4f} Val={val_metrics['val_wacc']:.4f} "
              f"Tail={val_metrics['tail_mean']:.4f}"
              + (f" | code_active={active_fraction:.3f}" if is_vq else ''))

        history.append({
            'method': spec.name, 'seed': seed, 'epoch': epoch,
            'L_task': round(avg_task, 4), 'L_vq': round(avg_vq, 4), 'L_use': round(avg_use, 4),
            'train_wacc': round(train_wacc, 4),
            'val_wacc': round(val_metrics['val_wacc'], 4),
            'val_auc': round(val_metrics['val_auc'], 4),
            'tail_mean': round(val_metrics['tail_mean'], 4),
            'code_active_fraction':
                round(active_fraction, 4) if active_fraction == active_fraction else None,
        })

        # Checkpoint selection. For non-VQ methods every epoch is eligible;
        # for VQ methods we wait until VQ is active.
        ckpt_eligible = (not is_vq) or use_vq_this_epoch
        if ckpt_eligible and val_metrics['val_wacc'] > best_wacc:
            best_wacc = val_metrics['val_wacc']
            best_pcr = val_metrics['per_class_recall']
            best_epoch = epoch
            patience_count = 0
            torch.save({
                'epoch': epoch,
                'best_wacc': best_wacc,
                'passive_state': passive.state_dict(),
                'active_state':  active.state_dict(),
                'method': spec.name,
                'passive_type': spec.passive_type,
                'passive_emb_dim': passive.passive_emb_dim,
                'comm_bits': spec.comm_bits,
                'seed': seed,
                'spec': asdict(spec),
            }, ckpt_path)
        elif ckpt_eligible:
            patience_count += 1
            if patience_count >= cfg.patience:
                print(f"  Early stop @ ep{epoch} (best {best_wacc:.4f} @ ep{best_epoch})")
                break

    save_metrics(
        f"{RESULTS_DIR}/history_{spec.name}_seed{seed}.json",
        {'history': history, 'method': spec.name, 'seed': seed},
    )
    result = {
        'method': spec.name,
        'seed': seed,
        'passive_type': spec.passive_type,
        'best_wacc': best_wacc,
        'best_epoch': best_epoch,
        'per_class_recall': best_pcr,
        'tail_mean': float(np.mean([best_pcr[c] for c in tail_classes])) if best_pcr else float('nan'),
        'comm_bits': spec.comm_bits,
        'ckpt_path': ckpt_path,
    }
    save_metrics(
        f"{RESULTS_DIR}/result_{spec.name}_seed{seed}.json", result,
    )

    del passive, active, opt_passive, opt_active
    torch.cuda.empty_cache()
    return result


print("train_v9 defined.")


train_v9 defined.


## §5 — Method specs (11 methods × 2 seeds = 22 runs)

Three "money comparisons" the grid is built around:

1. **H_vq_M16 vs S_sign_quant vs S_rand_sign** — all 128 bits. Tests whether learned
   VQ coding strictly dominates sign coding at matched budget, and whether learning
   the projection matters under sign quantisation.
2. **H_vq_K256 vs H_vq_no_kmeans** — same hyperparams, only codebook initialisation
   differs. Tests how load-bearing the K-means warmup is.
3. **A_proj_vfl SSIM vs A_plain_vfl SSIM** — first-time measurement of whether
   dimensionality reduction alone gives privacy.

### VQ communication-bits protocol (assumption made by every PQ/VQ-VAE paper)

For VQ methods we report `comm_bits = M · log2(K)` per sample (e.g. 64 bits for
K=256, M=8). This counts **per-sample inference bits** under the standard
deployment protocol:

- **Training time** (what this notebook does): codebooks are jointly learned;
  passive transmits decoded vectors `q ∈ R^D` for STE backprop.
- **Deployment time** (the regime the bit count refers to): codebooks are
  frozen and copied to the active side **once** (a one-time cost of
  `M · K · subdim · 32` bits — e.g. 1,048,576 bits / 128 KiB for K=256, M=8).
  Per-sample, the
  passive transmits only the **indices** (M · log2(K) bits) and the active
  reconstructs `q` via lookup. This is functionally equivalent to what the
  notebook simulates, because the argmin → lookup pipeline is deterministic
  given a fixed codebook.

If your protocol does not assume codebook sharing, add the codebook-transfer
cost amortised over your inference batch size to the reported bits. The
**relative** ordering between methods is unaffected.

Bit-count formulas:
- Continuous (A_plain, A_proj): float32 → 32 bits × dim
- Sign (S_sign_quant, S_rand_sign): 1 bit × dim = 128
- VQ (H_vq_*): M · log2(K)


In [15]:
@dataclass
class MethodSpec:
    name: str
    passive_type: str           # 'plain' | 'proj' | 'sign' | 'rand_sign' | 'vq'
    comm_bits: int
    # VQ-only (ignored otherwise)
    vq_codebook_size: int = 256
    vq_num_subspaces: int = 8
    vq_commitment_weight: float = 0.25
    vq_use_kmeans_init: bool = True
    # Per-method training epoch override (default matches V9Config.stage1_epochs)
    train_epochs: int = 40
    # Reconstruction stage: epoch budget (30 default)
    invernet_epochs: int = 50


METHOD_SPECS: List[MethodSpec] = [
    # --- Continuous baselines ---
    MethodSpec('A_plain_vfl',      'plain',     comm_bits=1280 * 32),  # 40960
    MethodSpec('A_proj_vfl',       'proj',      comm_bits=128 * 32),   # 4096
    # --- Sign baselines (128 bits each) ---
    MethodSpec('S_rand_sign',      'rand_sign', comm_bits=128),
    MethodSpec('S_sign_quant',     'sign',      comm_bits=128),
    # --- VQ K sweep ---
    MethodSpec('H_vq_K64',         'vq',        comm_bits=int(8 * math.log2(64)),       # 48
               vq_codebook_size=64,  vq_num_subspaces=8,  vq_commitment_weight=0.25),
    MethodSpec('H_vq_K256',        'vq',        comm_bits=int(8 * math.log2(256)),      # 64
               vq_codebook_size=256, vq_num_subspaces=8,  vq_commitment_weight=0.25,
               train_epochs=40),
    # --- VQ init ablation: same as K256 but no K-means seeding ---
    MethodSpec('H_vq_no_kmeans',   'vq',        comm_bits=int(8 * math.log2(256)),      # 64
               vq_codebook_size=256, vq_num_subspaces=8,  vq_commitment_weight=0.25,
               vq_use_kmeans_init=False, train_epochs=40),
    # --- VQ M sweep ---
    MethodSpec('H_vq_M4',          'vq',        comm_bits=int(4 * math.log2(256)),      # 32
               vq_codebook_size=256, vq_num_subspaces=4,  vq_commitment_weight=0.25,
               train_epochs=40),
    MethodSpec('H_vq_M16',         'vq',        comm_bits=int(16 * math.log2(256)),     # 128
               vq_codebook_size=256, vq_num_subspaces=16, vq_commitment_weight=0.25,
               train_epochs=40),
    # --- Commitment weight ablation ---
    MethodSpec('H_vq_commit_low',  'vq',        comm_bits=int(8 * math.log2(256)),      # 64
               vq_codebook_size=256, vq_num_subspaces=8,  vq_commitment_weight=0.1,
               train_epochs=40),
    MethodSpec('H_vq_commit_high', 'vq',        comm_bits=int(8 * math.log2(256)),      # 64
               vq_codebook_size=256, vq_num_subspaces=8,  vq_commitment_weight=0.5,
               train_epochs=40),
]

print(f"Defined {len(METHOD_SPECS)} method specs:")
print(f"{'name':<20s} {'type':<11s} {'bits':>6s}  vq_specifics")
for s in METHOD_SPECS:
    extra = ''
    if s.passive_type == 'vq':
        extra = (f"K={s.vq_codebook_size:<5d} M={s.vq_num_subspaces:<2d} "
                 f"beta={s.vq_commitment_weight}  kmeans={s.vq_use_kmeans_init}")
    print(f"  {s.name:<18s} {s.passive_type:<11s} {s.comm_bits:>6d}  {extra}")

# Persist the grid
with open(f"{RESULTS_DIR}/method_specs.json", 'w') as f:
    json.dump([asdict(s) for s in METHOD_SPECS], f, indent=2)

Defined 11 method specs:
name                 type          bits  vq_specifics
  A_plain_vfl        plain        40960  
  A_proj_vfl         proj          4096  
  S_rand_sign        rand_sign      128  
  S_sign_quant       sign           128  
  H_vq_K64           vq              48  K=64    M=8  beta=0.25  kmeans=True
  H_vq_K256          vq              64  K=256   M=8  beta=0.25  kmeans=True
  H_vq_no_kmeans     vq              64  K=256   M=8  beta=0.25  kmeans=False
  H_vq_M4            vq              32  K=256   M=4  beta=0.25  kmeans=True
  H_vq_M16           vq             128  K=256   M=16 beta=0.25  kmeans=True
  H_vq_commit_low    vq              64  K=256   M=8  beta=0.1  kmeans=True
  H_vq_commit_high   vq              64  K=256   M=8  beta=0.5  kmeans=True


In [16]:
# ============================================================================
# Stage A — Train all (method, seed) pairs.
# Each run skipped if its checkpoint already exists.
# ============================================================================
def run_stage_A(method_specs=METHOD_SPECS, seeds=None, force_retrain=False):
    seeds = seeds or list(CFG.seeds)
    completed, skipped, failed = [], [], []
    for spec in method_specs:
        for seed in seeds:
            ckpt_path = f"{CKPT_DIR}/{spec.name}_seed{seed}.pt"
            if (not force_retrain) and os.path.exists(ckpt_path):
                print(f"[skip] {spec.name} seed={seed}: ckpt exists at {ckpt_path}")
                skipped.append((spec.name, seed))
                continue
            try:
                t0 = time.time()
                _ = train_v9(spec, seed=seed)
                print(f"[done] {spec.name} seed={seed}: {time.time() - t0:.0f}s")
                completed.append((spec.name, seed))
            except Exception as e:
                print(f"[FAIL] {spec.name} seed={seed}: {e}")
                traceback.print_exc()
                failed.append((spec.name, seed, str(e)))
    print(f"\nStage A summary: completed={len(completed)} skipped={len(skipped)} failed={len(failed)}")
    return {'completed': completed, 'skipped': skipped, 'failed': failed}


# Stage A auto-runs when RUN_STAGE_A=True (top of cell 3). On Kaggle, this likely
# needs 2 sessions; the second session will skip already-completed runs because
# their checkpoints exist.
if RUN_STAGE_A:
    stage_a_summary = run_stage_A()
else:
    print("[skip] Stage A disabled (RUN_STAGE_A=False)")
    stage_a_summary = None


[skip] A_plain_vfl seed=42: ckpt exists at /kaggle/working/results_d1v9/checkpoints/A_plain_vfl_seed42.pt
[skip] A_plain_vfl seed=43: ckpt exists at /kaggle/working/results_d1v9/checkpoints/A_plain_vfl_seed43.pt
[skip] A_proj_vfl seed=42: ckpt exists at /kaggle/working/results_d1v9/checkpoints/A_proj_vfl_seed42.pt
[skip] A_proj_vfl seed=43: ckpt exists at /kaggle/working/results_d1v9/checkpoints/A_proj_vfl_seed43.pt
[skip] S_rand_sign seed=42: ckpt exists at /kaggle/working/results_d1v9/checkpoints/S_rand_sign_seed42.pt
[skip] S_rand_sign seed=43: ckpt exists at /kaggle/working/results_d1v9/checkpoints/S_rand_sign_seed43.pt
[skip] S_sign_quant seed=42: ckpt exists at /kaggle/working/results_d1v9/checkpoints/S_sign_quant_seed42.pt
[skip] S_sign_quant seed=43: ckpt exists at /kaggle/working/results_d1v9/checkpoints/S_sign_quant_seed43.pt
[skip] H_vq_K64 seed=42: ckpt exists at /kaggle/working/results_d1v9/checkpoints/H_vq_K64_seed42.pt
[skip] H_vq_K64 seed=43: ckpt exists at /kaggle/work

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Ep 01/40 (cont.) | L_task=2.0454 L_vq=0.0000 L_use=0.0000 | Tr=0.1755 Val=0.2274 Tail=0.1163 | code_active=nan


Ep 02/40 (cont.) | L_task=1.9291 L_vq=0.0000 L_use=0.0000 | Tr=0.2832 Val=0.3387 Tail=0.2358 | code_active=nan


Ep 03/40 (cont.) | L_task=1.6814 L_vq=0.0000 L_use=0.0000 | Tr=0.3841 Val=0.4293 Tail=0.3678 | code_active=nan
  [epoch 4] running K-means++ codebook seeding ...


...

Ep 29/40 (VQ on) | L_task=0.5529 L_vq=0.4112 L_use=-4.6363 | Tr=0.8131 Val=0.7445 Tail=0.7380 | code_active=0.956


Ep 30/40 (VQ on) | L_task=0.5595 L_vq=0.4028 L_use=-4.6312 | Tr=0.8089 Val=0.7680 Tail=0.7791 | code_active=0.960


Ep 31/40 (VQ on) | L_task=0.5409 L_vq=0.3984 L_use=-4.6334 | Tr=0.8231 Val=0.7637 Tail=0.7770 | code_active=0.961


Ep 32/40 (VQ on) | L_task=0.5314 L_vq=0.3920 L_use=-4.6304 | Tr=0.8278 Val=0.7696 Tail=0.7828 | code_active=0.961


Ep 33/40 (VQ on) | L_task=0.5157 L_vq=0.3884 L_use=-4.6283 | Tr=0.8304 Val=0.7634 Tail=0.7698 | code_active=0.959


In [ ]:
import shutil
from IPython.display import FileLink

# 1. Zip the entire checkpoints folder
# Format: shutil.make_archive('desired_zip_name', 'zip', 'path_to_folder')
shutil.make_archive('checkpoints_backup', 'zip', 'results_d1v9/checkpoints')

# 2. Generate a clickable download link
FileLink('checkpoints_backup.zip')

In [ ]:
print("hello")

## §6 — Stage B: Reconstruction privacy evaluation

InverNetV9 (residual blocks at the bottleneck + final refinement conv + MSE+0.1·LPIPS
loss) is trained on the *transmitted representation* (whatever the passive client
emits) of the train fold and evaluated on the val fold. SSIM and LPIPS are reported.

50 epochs for all methods (final setting — uniform across all methods).

In [ ]:
# -------- InverNetV9: improved attacker --------
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, 1, 1)
        self.b2 = nn.BatchNorm2d(ch)
    def forward(self, x):
        return F.relu(x + self.b2(self.c2(F.relu(self.b1(self.c1(x))))))


class InverNetV9(nn.Module):
    '''Stronger InverNet: residual bottleneck + refinement + LPIPS-augmented loss.

    The convolutional decoder (ResBlocks + upsampler + refine) is identical across
    all methods. The fc layer adapts the input dimension to the 256-ch 4x4 spatial
    feature map and naturally scales with in_dim — a stronger attacker for richer
    representations, which is the correct choice for a privacy evaluation.
    '''
    def __init__(self, in_dim: int, out_resolution: int = 64):
        super().__init__()
        self.out_resolution = out_resolution
        self.fc = nn.Linear(in_dim, 256 * 4 * 4)
        self.res_blocks = nn.Sequential(ResBlock(256), ResBlock(256))
        self.upsampler = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),  # 8
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),  # 16
            nn.ConvTranspose2d( 64,  32, 4, 2, 1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True),  # 32
            nn.ConvTranspose2d( 32,   3, 4, 2, 1),                                              # 64
        )
        self.refine = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 3, 3, 1, 1), nn.Tanh(),
        )

    def forward(self, x):
        h = self.fc(x).view(-1, 256, 4, 4)
        h = self.res_blocks(h)
        h = self.upsampler(h)
        return self.refine(h)


print("InverNetV9 defined.")


In [ ]:
INV_RESOLUTION = 64

def _build_passive_for_load(spec_dict, seed: int):
    '''Reconstruct the passive client class from a saved spec dict.'''
    # Convert dict back to MethodSpec-ish for build_passive
    s = MethodSpec(
        name=spec_dict['name'],
        passive_type=spec_dict['passive_type'],
        comm_bits=spec_dict['comm_bits'],
        vq_codebook_size=spec_dict.get('vq_codebook_size', 256),
        vq_num_subspaces=spec_dict.get('vq_num_subspaces', 8),
        vq_commitment_weight=spec_dict.get('vq_commitment_weight', 0.25),
        vq_use_kmeans_init=spec_dict.get('vq_use_kmeans_init', True),
        invernet_epochs=spec_dict.get('invernet_epochs', 50),
    )
    return build_passive(s, seed=seed), s


def reconstruction_attack_v9(ckpt_path: str, epochs: int = 50,
                             target_resolution: int = INV_RESOLUTION,
                             save_grid_path: str = None,
                             grid_indices: List[int] = None) -> Dict[str, Any]:
    '''Train InverNetV9 to invert the transmitted rep -> low-res image.

    Returns dict with ssim_mean/std, lpips_mean/std, feat_dim. Never raises;
    on failure returns {'error': ..., 'skipped': True}.
    '''
    try:
        import lpips as lpips_module
        from skimage.metrics import structural_similarity as ssim_fn
    except Exception as e:
        return {'error': f'lpips/skimage import failed: {e}', 'skipped': True}

    try:
        ckpt = _torch_load(ckpt_path)
        spec_dict = ckpt['spec']
        seed = ckpt['seed']
        passive, _ = _build_passive_for_load(spec_dict, seed=seed)
        passive = passive.to(DEVICE)
        miss, unexp = passive.load_state_dict(ckpt['passive_state'], strict=True)
        assert len(miss) == 0 and len(unexp) == 0, f"load mismatch m={miss} u={unexp}"
        passive.eval()
        # VQ methods need quantiser active for inversion (this is the operating regime)
        use_quantizer = (spec_dict['passive_type'] == 'vq')

        # IMPORTANT: target_tf must view the SAME spatial extent as the encoder.
        # The encoder feed (val_transforms) does CenterCrop(input_size). If the
        # target uses Resize() on the full image, InverNet is asked to recover
        # corner regions the encoder never saw — biasing every SSIM downward and
        # overstating privacy. Mirror the crop, then resize to the recon resolution.
        target_tf = transforms.Compose([
            transforms.CenterCrop(CFG.input_size),
            transforms.Resize((target_resolution, target_resolution)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),  # to [-1, 1]
        ])

        class PairDataset(Dataset):
            def __init__(self, images, feat_tf, target_tf):
                self.images = images
                self.feat_tf = feat_tf
                self.target_tf = target_tf
            def __len__(self): return len(self.images)
            def __getitem__(self, idx):
                pil = Image.fromarray(self.images[idx])
                return self.feat_tf(pil), self.target_tf(pil)

        tr_ds  = PairDataset(all_images[train_ind], val_transforms, target_tf)
        val_ds = PairDataset(all_images[val_ind],   val_transforms, target_tf)
        tr_loader  = DataLoader(tr_ds,  batch_size=64, shuffle=True,
                                num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False,
                                num_workers=CFG.num_workers, pin_memory=True)

        with torch.no_grad():
            sample_img, _ = next(iter(tr_loader))
            sample_img = sample_img.to(DEVICE)
            emb, _, _, _ = passive(sample_img[:1], use_quantizer=use_quantizer)
            feat_dim = emb.shape[1]

        decoder = InverNetV9(feat_dim, target_resolution).to(DEVICE)
        opt = optim.Adam(decoder.parameters(), lr=1e-3, weight_decay=1e-5)
        sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
        mse_crit = nn.MSELoss()

        # LPIPS as differentiable perceptual loss term, frozen
        lpips_loss_net = lpips_module.LPIPS(net='alex', verbose=False).to(DEVICE)
        for p in lpips_loss_net.parameters(): p.requires_grad = False
        lpips_loss_net.eval()

        for ep in range(1, epochs + 1):
            decoder.train()
            ep_l = 0.0; n = 0
            for img_feat, img_tgt in tr_loader:
                img_feat = img_feat.to(DEVICE); img_tgt = img_tgt.to(DEVICE)
                with torch.no_grad():
                    emb, _, _, _ = passive(img_feat, use_quantizer=use_quantizer)
                recon = decoder(emb)
                l_mse = mse_crit(recon, img_tgt)
                l_lpips = lpips_loss_net(recon, img_tgt).mean()
                loss = l_mse + 0.1 * l_lpips
                opt.zero_grad(); loss.backward(); opt.step()
                ep_l += float(loss.item()); n += 1
            sched.step()
            if ep == 1 or ep % 5 == 0 or ep == epochs:
                print(f"    [InverNet] ep {ep:02d}/{epochs}  loss={ep_l/max(1,n):.4f}")

        # Evaluation
        decoder.eval()
        ssim_scores, lpips_scores = [], []
        with torch.no_grad():
            for img_feat, img_tgt in val_loader:
                img_feat = img_feat.to(DEVICE); img_tgt = img_tgt.to(DEVICE)
                emb, _, _, _ = passive(img_feat, use_quantizer=use_quantizer)
                recon = decoder(emb)
                lp = lpips_loss_net(recon, img_tgt).squeeze().detach().cpu().numpy()
                if lp.ndim == 0:
                    lp = np.array([float(lp)])
                lpips_scores.extend(lp.tolist())
                r_np = ((recon.clamp(-1, 1) + 1) / 2).permute(0, 2, 3, 1).cpu().numpy()
                t_np = ((img_tgt.clamp(-1, 1) + 1) / 2).permute(0, 2, 3, 1).cpu().numpy()
                for b in range(r_np.shape[0]):
                    try:
                        s = ssim_fn(t_np[b], r_np[b], channel_axis=2, data_range=1.0,
                                    gaussian_weights=True, sigma=1.5,
                                    use_sample_covariance=False)
                        ssim_scores.append(float(s))
                    except Exception:
                        pass

        # Optional aligned visual grid
        if save_grid_path is not None and grid_indices is not None:
            _save_grid_png(passive, decoder, grid_indices, save_grid_path,
                           target_tf=target_tf, use_quantizer=use_quantizer)

        return {
            'ssim_mean': float(np.mean(ssim_scores)) if ssim_scores else float('nan'),
            'ssim_std':  float(np.std(ssim_scores))  if ssim_scores else float('nan'),
            'lpips_mean': float(np.mean(lpips_scores)) if lpips_scores else float('nan'),
            'lpips_std':  float(np.std(lpips_scores))  if lpips_scores else float('nan'),
            'n_val_samples': len(ssim_scores),
            'feat_dim': int(feat_dim),
            'invernet_epochs': int(epochs),
            'skipped': False,
        }
    except Exception as e:
        return {'error': str(e), 'trace': traceback.format_exc(), 'skipped': True}


@torch.no_grad()
def _save_grid_png(passive, decoder, sample_indices: List[int], out_path: str,
                   target_tf, use_quantizer: bool):
    '''Save a multi-row PNG: rows=samples, cols=[orig | recon | diff].'''
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    passive.eval(); decoder.eval()
    n = len(sample_indices)
    fig, axes = plt.subplots(n, 3, figsize=(6, 2 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for row, idx in enumerate(sample_indices):
        pil = Image.fromarray(all_images[idx])
        feat_in = val_transforms(pil).unsqueeze(0).to(DEVICE)
        tgt_in  = target_tf(pil).unsqueeze(0).to(DEVICE)
        emb, _, _, _ = passive(feat_in, use_quantizer=use_quantizer)
        recon = decoder(emb)
        r_np = ((recon.clamp(-1, 1) + 1) / 2).squeeze(0).permute(1, 2, 0).cpu().numpy()
        t_np = ((tgt_in.clamp(-1, 1) + 1) / 2).squeeze(0).permute(1, 2, 0).cpu().numpy()
        diff = np.abs(r_np - t_np)
        for c, (img, title) in enumerate([
            (t_np, 'orig'), (r_np, 'recon'), (diff / max(diff.max(), 1e-6), 'diff'),
        ]):
            axes[row, c].imshow(np.clip(img, 0, 1))
            axes[row, c].set_axis_off()
            if row == 0: axes[row, c].set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


print("Reconstruction attacker defined.")

In [ ]:
# Pick 6 sample indices that we use across ALL methods, to keep visual
# comparisons strictly aligned. Cover at least 2 tail classes (DF, VASC).
def _pick_grid_indices(n_per_class_tail=2, n_per_class_head=2):
    '''Choose val-set indices: 2 from each of the 4 rarest classes (8 total), trimmed to 6.'''
    val_labels = label_ints[val_ind]
    chosen = []
    rng = np.random.RandomState(20260429)  # deterministic
    # First, tail classes: 1 each
    for c in tail_classes:
        cands = np.where(val_labels == c)[0]
        if len(cands) > 0:
            chosen.append(int(val_ind[cands[rng.randint(0, len(cands))]]))
    # Then a couple of head classes
    head_classes = [c for c in range(NUM_CLASSES) if c not in tail_classes]
    for c in head_classes[:2]:
        cands = np.where(val_labels == c)[0]
        if len(cands) > 0:
            chosen.append(int(val_ind[cands[rng.randint(0, len(cands))]]))
    chosen = chosen[:6]
    return chosen


GRID_INDICES = _pick_grid_indices()
print(f"Aligned grid indices ({len(GRID_INDICES)}): {GRID_INDICES}")
print(f"  classes: {[CLASS_NAMES[label_ints[i]] for i in GRID_INDICES]}")
with open(f"{RESULTS_DIR}/grid_indices.json", 'w') as f:
    json.dump({'indices': GRID_INDICES,
               'classes':  [CLASS_NAMES[label_ints[i]] for i in GRID_INDICES]}, f, indent=2)


In [ ]:
# ============================================================================
# Stage B driver — run reconstruction attack on every checkpoint.
# Skips already-evaluated (recon JSON exists).
# ============================================================================
def run_stage_B(method_specs=METHOD_SPECS, seeds=None, force_reeval=False):
    seeds = seeds or list(CFG.seeds)
    results = {}
    for spec in method_specs:
        for seed in seeds:
            ckpt_path = f"{CKPT_DIR}/{spec.name}_seed{seed}.pt"
            recon_path = f"{RESULTS_DIR}/recon_{spec.name}_seed{seed}.json"
            grid_path  = f"{FIG_DIR}/recon_grid_{spec.name}_seed{seed}.png"
            if not os.path.exists(ckpt_path):
                print(f"[miss ckpt] {spec.name} seed={seed}")
                continue
            if (not force_reeval) and os.path.exists(recon_path):
                print(f"[skip] {spec.name} seed={seed}: recon json exists")
                with open(recon_path) as f:
                    results[(spec.name, seed)] = json.load(f)
                continue
            print(f"\n>>> Reconstruction attack: {spec.name} seed={seed} "
                  f"(epochs={spec.invernet_epochs})")
            t0 = time.time()
            res = reconstruction_attack_v9(
                ckpt_path,
                epochs=spec.invernet_epochs,
                save_grid_path=grid_path,
                grid_indices=GRID_INDICES,
            )
            res['method']  = spec.name
            res['seed']    = seed
            res['took_s']  = time.time() - t0
            save_metrics(recon_path, res)
            results[(spec.name, seed)] = res
            if res.get('skipped'):
                print(f"  [skipped] {res.get('error')}")
            else:
                print(f"  SSIM={res['ssim_mean']:.4f}+-{res['ssim_std']:.4f}  "
                      f"LPIPS={res['lpips_mean']:.4f}+-{res['lpips_std']:.4f}  "
                      f"({res['took_s']:.0f}s)")
    return results


if RUN_STAGE_B:
    stage_b_results = run_stage_B()
else:
    print("[skip] Stage B disabled (RUN_STAGE_B=False)")
    stage_b_results = {}


## §7-§8 — Stage C: aggregation + figures

Reads all `result_*.json` (utility) and `recon_*.json` (privacy), produces:
- combined results table (CSV + JSON)
- Pareto curve (WACC vs SSIM, coloured by family, annotated with bits)
- bits vs SSIM and bits vs WACC plots
- multi-row visual reconstruction comparison (one row per method, aligned 6 samples)


In [ ]:
def _mean_std(vals):
    arr = np.array([v for v in vals if v is not None and not np.isnan(v)])
    if arr.size == 0: return float('nan'), float('nan')
    return float(arr.mean()), float(arr.std())


def aggregate_results(method_specs=METHOD_SPECS, seeds=None) -> pd.DataFrame:
    seeds = seeds or list(CFG.seeds)
    rows = []
    for spec in method_specs:
        wacc_vals, tail_vals = [], []
        ssim_vals, lpips_vals = [], []
        for seed in seeds:
            try:
                with open(f"{RESULTS_DIR}/result_{spec.name}_seed{seed}.json") as f:
                    r = json.load(f)
                wacc_vals.append(r.get('best_wacc'))
                tail_vals.append(r.get('tail_mean'))
            except FileNotFoundError:
                pass
            try:
                with open(f"{RESULTS_DIR}/recon_{spec.name}_seed{seed}.json") as f:
                    rc = json.load(f)
                if not rc.get('skipped', False):
                    ssim_vals.append(rc.get('ssim_mean'))
                    lpips_vals.append(rc.get('lpips_mean'))
            except FileNotFoundError:
                pass
        wacc_m, wacc_s   = _mean_std(wacc_vals)
        tail_m, tail_s   = _mean_std(tail_vals)
        ssim_m, ssim_s   = _mean_std(ssim_vals)
        lpips_m, lpips_s = _mean_std(lpips_vals)
        rows.append({
            'method': spec.name,
            'family': spec.passive_type,
            'comm_bits': spec.comm_bits,
            'wacc_mean':  wacc_m,  'wacc_std':  wacc_s,
            'tail_mean':  tail_m,  'tail_std':  tail_s,
            'ssim_mean':  ssim_m,  'ssim_std':  ssim_s,
            'lpips_mean': lpips_m, 'lpips_std': lpips_s,
            'n_seeds': len(wacc_vals),
        })
    df = pd.DataFrame(rows)
    df.to_csv(f"{RESULTS_DIR}/results_table.csv", index=False)
    with open(f"{RESULTS_DIR}/results_table.json", 'w') as f:
        json.dump(rows, f, indent=2, default=str)
    return df


df_results = aggregate_results()
print("\n========= V9 Final Results Table =========")
with pd.option_context('display.float_format', '{:.4f}'.format,
                       'display.max_columns', None,
                       'display.width', 160):
    print(df_results.to_string(index=False))


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FAMILY_COLORS = {
    'plain':     '#444444',
    'proj':      '#888888',
    'sign':      '#1f77b4',
    'rand_sign': '#aec7e8',
    'vq':        '#d62728',
}

def plot_pareto(df: pd.DataFrame, out_path: str):
    fig, ax = plt.subplots(figsize=(8, 6))
    for _, r in df.iterrows():
        if pd.isna(r['ssim_mean']) or pd.isna(r['wacc_mean']): continue
        c = FAMILY_COLORS.get(r['family'], '#000000')
        ax.errorbar(r['ssim_mean'], r['wacc_mean'],
                    xerr=r.get('ssim_std', 0) or 0,
                    yerr=r.get('wacc_std', 0) or 0,
                    fmt='o', color=c, capsize=3, markersize=8)
        ax.annotate(f"{r['method']}\n({int(r['comm_bits'])}b)",
                    (r['ssim_mean'], r['wacc_mean']),
                    fontsize=8, xytext=(6, 4), textcoords='offset points')
    ax.set_xlabel('Reconstruction SSIM (higher = worse privacy)')
    ax.set_ylabel('Validation WACC (higher = better utility)')
    ax.set_title('Privacy-utility Pareto: WACC vs SSIM (annotations = comm bits)')
    ax.grid(alpha=0.3)
    # Family legend
    handles = [plt.Line2D([], [], marker='o', color=c, linestyle='', label=fam)
               for fam, c in FAMILY_COLORS.items()]
    ax.legend(handles=handles, title='family', loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=140)
    plt.close(fig)


def plot_bits_vs(df: pd.DataFrame, metric: str, ylabel: str, out_path: str,
                 invert_y: bool = False):
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, r in df.iterrows():
        v = r[f'{metric}_mean']
        if pd.isna(v): continue
        c = FAMILY_COLORS.get(r['family'], '#000000')
        ax.errorbar(r['comm_bits'], v,
                    yerr=r.get(f'{metric}_std', 0) or 0,
                    fmt='o', color=c, capsize=3, markersize=8)
        ax.annotate(r['method'], (r['comm_bits'], v),
                    fontsize=8, xytext=(6, 4), textcoords='offset points')
    ax.set_xscale('log')
    ax.set_xlabel('Communication bits per sample (log scale)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3, which='both')
    if invert_y: ax.invert_yaxis()
    handles = [plt.Line2D([], [], marker='o', color=c, linestyle='', label=fam)
               for fam, c in FAMILY_COLORS.items()]
    ax.legend(handles=handles, title='family', loc='best')
    fig.tight_layout()
    fig.savefig(out_path, dpi=140)
    plt.close(fig)


if RUN_STAGE_C:
    plot_pareto(df_results, f"{FIG_DIR}/pareto_wacc_vs_ssim.png")
    plot_bits_vs(df_results, 'ssim',  'Reconstruction SSIM (higher = worse privacy)',
                 f"{FIG_DIR}/bits_vs_ssim.png")
    plot_bits_vs(df_results, 'wacc',  'Validation WACC',
                 f"{FIG_DIR}/bits_vs_wacc.png")
    plot_bits_vs(df_results, 'lpips', 'Reconstruction LPIPS (lower = worse privacy)',
                 f"{FIG_DIR}/bits_vs_lpips.png", invert_y=True)
    print(f"Saved Pareto + bits-vs-* plots to {FIG_DIR}")
else:
    print("[skip] Stage C plotting disabled (RUN_STAGE_C=False)")


In [ ]:
# ----------------------------------------------------------------------
# Multi-row visual reconstruction comparison.
# Stack per-method recon grids (saved during Stage B) into one figure for the paper.
# ----------------------------------------------------------------------
def build_visual_comparison(method_specs=METHOD_SPECS, seed: int = None,
                             out_path: str = None):
    seed = seed or CFG.seeds[0]
    out_path = out_path or f"{FIG_DIR}/visual_comparison_seed{seed}.png"
    n = len(GRID_INDICES)
    methods_with_grid = []
    for spec in method_specs:
        gp = f"{FIG_DIR}/recon_grid_{spec.name}_seed{seed}.png"
        if os.path.exists(gp):
            methods_with_grid.append((spec.name, gp))
    if not methods_with_grid:
        print("No recon grids found; run Stage B first.")
        return None

    fig, axes = plt.subplots(len(methods_with_grid), 1,
                              figsize=(7, 2.0 * len(methods_with_grid)))
    if len(methods_with_grid) == 1:
        axes = [axes]
    for ax, (name, gp) in zip(axes, methods_with_grid):
        img = plt.imread(gp)
        ax.imshow(img)
        ax.set_axis_off()
        ax.set_title(name, fontsize=10, loc='left')
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    print(f"Saved visual comparison: {out_path}")
    return out_path


if RUN_STAGE_C:
    build_visual_comparison()


In [ ]:
# ----------------------------------------------------------------------
# Final summary printout
# ----------------------------------------------------------------------
print("=" * 90)
print("V9 RUN SUMMARY")
print("=" * 90)
print(f"Methods: {len(METHOD_SPECS)}  ×  Seeds: {len(CFG.seeds)}  =  "
      f"{len(METHOD_SPECS) * len(CFG.seeds)} total runs")
print(f"Results dir: {RESULTS_DIR}")
print(f"Checkpoints: {CKPT_DIR}")
print(f"Figures:     {FIG_DIR}")
print()
print("--- Headline comparisons ---")
df = df_results.set_index('method')
def _row(m):
    if m not in df.index: return f"{m}: not run"
    r = df.loc[m]
    return (f"{m:<20s}  WACC={r['wacc_mean']:.4f}+-{r['wacc_std']:.4f}  "
            f"SSIM={r['ssim_mean']:.4f}+-{r['ssim_std']:.4f}  "
            f"LPIPS={r['lpips_mean']:.4f}+-{r['lpips_std']:.4f}  "
            f"bits={int(r['comm_bits'])}")

print("[A] 128-bit comparison (the apples-to-apples)")
for m in ['S_rand_sign', 'S_sign_quant', 'H_vq_M16']:
    print('  ' + _row(m))

print("\n[B] K-means init ablation")
for m in ['H_vq_K256', 'H_vq_no_kmeans']:
    print('  ' + _row(m))

print("\n[C] Privacy from dim reduction alone")
for m in ['A_plain_vfl', 'A_proj_vfl']:
    print('  ' + _row(m))

print("\n[D] Commitment weight ablation")
for m in ['H_vq_commit_low', 'H_vq_K256', 'H_vq_commit_high']:
    print('  ' + _row(m))

print("\nDone.")
